# Databricks Genie via Amazon Bedrock AgentCore Gateway (MCP)

Expose a [Databricks Genie](https://docs.databricks.com/en/genie/index.html) space as a governed MCP tool to Amazon Bedrock agents through [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html). Bedrock agents ask plain-English business questions; Genie returns lakehouse-native SQL answers with Unity Catalog governance, identity, and lineage preserved end-to-end.

![Architecture](images/architecture.png)


## About this sample

This sample adds the **Genie** surface — Databricks' natural-language analytics layer grounded in Unity Catalog Trusted Assets — so Bedrock agents can ask business questions and receive grounded, governed SQL answers without a custom NL-to-SQL chain. It complements the two existing Databricks integrations in this folder:

- [`databricks-dbsql-agentcore-gateway`](../databricks-dbsql-agentcore-gateway) — Databricks SQL MCP via Gateway with M2M auth
- [`databricks-dbsql-per-user-delegation`](../databricks-dbsql-per-user-delegation) — Per-user delegation via RFC 8693 token exchange


## Databricks Genie Overview

[Databricks Genie](https://docs.databricks.com/en/genie/index.html) is a natural-language analytics surface grounded in [Unity Catalog](https://docs.databricks.com/en/data-governance/unity-catalog/index.html). It uses Trusted Assets — curated metrics, sample queries, and table descriptions — to ground SQL generation in business semantics. A Genie *space* is the unit you point at a specific data domain (finance, marketing, operations, etc.).

Databricks ships [managed MCP servers](https://docs.databricks.com/en/generative-ai/mcp/managed-mcp.html) that expose Genie spaces via a Model Context Protocol endpoint at `/api/2.0/mcp/genie/{space_id}`. The endpoint exposes a `query_genie` tool that takes a natural-language question and returns the SQL, narrative, and result set.


## Amazon Bedrock AgentCore Gateway Overview

[Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html) is a managed MCP gateway in front of your tool surfaces. For this sample it handles:

- **Inbound auth** — agent → gateway authorization (via AgentCore Identity, optionally fronted by Amazon Cognito)
- **Outbound auth** — Databricks OAuth2 M2M credentials, registered via `CreateOauth2CredentialProvider`, retrieved by Gateway at tool-invocation time
- **Routing** — forwards `tools/call` MCP requests to the Databricks-managed Genie MCP endpoint
- **Audit** — emits CloudWatch traces for every tool invocation


## Setting up Databricks Credentials

To authenticate AgentCore Gateway with your Databricks workspace, you need a **service principal** with an OAuth secret. This enables machine-to-machine (M2M) authentication using the OAuth2 client credentials flow.

### Step 1: Create a Service Principal

1. In your Databricks workspace, go to **Settings** → **Identity and access** → **Service principals**
2. Click **Add service principal** and provide a name (e.g., `agentcore-genie-gateway`)
3. After creation, note the **Application ID** — this is your `client_id`

### Step 2: Generate an OAuth Secret

1. Select the service principal you just created
2. Go to the **Secrets** tab and click **Generate secret**
3. Copy the **Secret** value immediately — it will not be shown again. This is your `client_secret`

### Step 3: Grant Genie Space + Unity Catalog Permissions

1. **Genie Space** — open your Genie space → **Settings** → grant the service principal **CAN RUN** on the space
2. **Unity Catalog tables** behind the Genie space — grant `USE CATALOG`, `USE SCHEMA`, `SELECT` to the service principal

For more details, see the [Databricks OAuth M2M documentation](https://docs.databricks.com/en/dev-tools/auth/oauth-m2m.html).


### Databricks Managed Genie MCP Server

Databricks managed MCP servers are available out of the box — no additional setup required beyond authentication. The Genie MCP server endpoint is:

```
https://<your-workspace-host>/api/2.0/mcp/genie/<genie-space-id>
```

Capture the **Genie space ID** from the workspace UI: Genie → your space → URL contains `/spaces/<id>`. For the full list of managed MCP servers, see the [Databricks managed MCP docs](https://docs.databricks.com/en/generative-ai/mcp/managed-mcp.html).


## Step 1: Configure Your Environment

Set your Databricks workspace host, service principal credentials, and Genie space ID. Replace the placeholder values below with your own.


In [ ]:
import os

# Databricks workspace URL (e.g., https://dbc-xxxxxxxx-xxxx.cloud.databricks.com)
DATABRICKS_HOST = ""

# Service principal credentials from the steps above
DATABRICKS_CLIENT_ID = ""      # Application ID of the service principal
DATABRICKS_CLIENT_SECRET = ""  # OAuth secret you generated

# Genie space ID (from the URL in your Databricks workspace)
GENIE_SPACE_ID = ""

# AWS region for AgentCore resources
REGION = "us-east-1"

assert DATABRICKS_HOST,        "Please set DATABRICKS_HOST"
assert DATABRICKS_CLIENT_ID,    "Please set DATABRICKS_CLIENT_ID"
assert DATABRICKS_CLIENT_SECRET,"Please set DATABRICKS_CLIENT_SECRET"
assert GENIE_SPACE_ID,          "Please set GENIE_SPACE_ID"


Install dependencies and create boto3 clients:


In [ ]:
%pip install --quiet boto3 bedrock-agentcore bedrock-agentcore-starter-toolkit strands-agents strands-agents-tools mcp pyyaml


In [ ]:
import json
import time
import logging

import boto3
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

sts = boto3.client('sts')
account_id = sts.get_caller_identity().get('Account')
agentcore = boto3.client('bedrock-agentcore-control', region_name=REGION)

print(f'AWS Account: {account_id}')
print(f'Region:      {REGION}')


## Step 2: Create or Reuse an AgentCore Gateway

Two options:

- **Option A** — Create a new gateway from scratch (run the cells in this section)
- **Option B** — Reuse an existing gateway by loading its configuration from `gateway_config.json`

If you already have a gateway deployed (e.g., from one of the sibling samples), skip to Option B.


### Option A: Create a New Gateway


In [ ]:
client = GatewayClient(region_name=REGION)
client.logger.setLevel(logging.INFO)

print('Creating Cognito authorizer (inbound auth)...')
cognito = client.create_oauth_authorizer_with_cognito('DatabricksGenieGateway')

print('Creating Gateway...')
gateway = client.create_mcp_gateway(
    name='DatabricksGenieGateway',
    role_arn=None,
    authorizer_config=cognito['authorizer_config'],
    enable_semantic_search=True,
)
client.fix_iam_permissions(gateway)

gateway_url = gateway['gatewayUrl']
gateway_id  = gateway['gatewayId']

print(f'Gateway URL: {gateway_url}')
print(f'Gateway ID:  {gateway_id}')
print('Waiting 30s for IAM propagation...')
time.sleep(30)


### Create Databricks OAuth2 Credential Provider (Outbound Auth)

AgentCore Identity manages the outbound OAuth2 credentials so the gateway can authenticate with Databricks on behalf of your agent. We create a credential provider that stores the service principal's client ID and secret, and points at the Databricks OIDC token endpoint.


In [ ]:
host = DATABRICKS_HOST.rstrip('/')
token_endpoint = f'{host}/oidc/v1/token'

print('Creating Databricks OAuth2 credential provider...')
cred_provider = agentcore.create_oauth2_credential_provider(
    name='databricks-genie-oauth',
    credentialProviderVendor='CustomOauth2',
    oauth2ProviderConfigInput={
        'customOauth2ProviderConfig': {
            'oauthDiscovery': {
                'authorizationServerMetadata': {
                    'issuer':                host,
                    'tokenEndpoint':         token_endpoint,
                    'authorizationEndpoint': token_endpoint,
                }
            },
            'clientId':     DATABRICKS_CLIENT_ID,
            'clientSecret': DATABRICKS_CLIENT_SECRET,
        }
    },
)

provider_arn = cred_provider['credentialProviderArn']
secret_arn   = cred_provider.get('secretArn') or cred_provider.get('clientSecretArn', {}).get('secretArn', '')
print(f'Credential provider ARN: {provider_arn}')


### Update Gateway Role Permissions

The gateway's IAM role needs three permissions to use the credential provider end-to-end: fetch workload access tokens, retrieve OAuth2 tokens from the credential provider, and read the stored secret.


In [ ]:
print('Updating gateway role permissions...')
gateway_details = agentcore.get_gateway(gatewayIdentifier=gateway_id)
role_arn  = gateway_details['roleArn']
role_name = role_arn.split('/')[-1]

iam = boto3.client('iam')
policy_doc = json.dumps({
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect':   'Allow',
            'Action':   'bedrock-agentcore:GetWorkloadAccessToken',
            'Resource': [
                f'arn:aws:bedrock-agentcore:{REGION}:*:workload-identity-directory/default',
                f'arn:aws:bedrock-agentcore:{REGION}:*:workload-identity-directory/default/workload-identity/DatabricksGenieGateway-*',
            ],
        },
        {
            'Effect':   'Allow',
            'Action':   'bedrock-agentcore:GetResourceOauth2Token',
            'Resource': provider_arn,
        },
        {
            'Effect':   'Allow',
            'Action':   'secretsmanager:GetSecretValue',
            'Resource': secret_arn,
        },
    ],
})

iam.put_role_policy(
    RoleName=role_name,
    PolicyName='DatabricksGenieOAuthAccess',
    PolicyDocument=policy_doc,
)
print(f'Updated role: {role_name}')
time.sleep(10)


### Add the Databricks Genie MCP Server as a Gateway Target

Register the Databricks-managed Genie MCP endpoint as a target on the gateway. The gateway will use the OAuth2 credential provider we just created to authenticate outbound requests to Databricks.


In [ ]:
mcp_url = f'{host}/api/2.0/mcp/genie/{GENIE_SPACE_ID}'

print(f'Adding Databricks Genie MCP server target: {mcp_url}')
target = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name='DatabricksGenie',
    description=f'Databricks Genie space {GENIE_SPACE_ID} as MCP tool',
    targetConfiguration={
        'mcp': {
            'mcpServer': {
                'endpoint': mcp_url,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            'credentialProviderType': 'OAUTH',
            'credentialProvider': {
                'oauthCredentialProvider': {
                    'providerArn': provider_arn,
                    'grantType':   'CLIENT_CREDENTIALS',
                    'scopes':      ['genie'],
                }
            },
        }
    ],
)

target_id = target['targetId']
print(f'Target ID: {target_id}')


Wait for the target to become ready, then synchronize the tool surface from Databricks:


In [ ]:
print('Waiting for target to be ready...')
for _ in range(24):
    t = agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    if t.get('status') not in ('Creating', 'Updating'):
        break
    time.sleep(5)
print(f'Target status: {t.get("status")}')

print('Synchronizing tools from Databricks...')
agentcore.synchronize_gateway_targets(
    gatewayIdentifier=gateway_id,
    targetIdList=[target_id],
)
print('Tools synchronized.')


### Save Configuration

Save the gateway + target identifiers so subsequent runs (or a deployed agent) can reuse them.


In [ ]:
config = {
    'gateway_id':     gateway_id,
    'gateway_url':    gateway_url,
    'target_id':      target_id,
    'provider_arn':   provider_arn,
    'genie_space_id': GENIE_SPACE_ID,
    'region':         REGION,
    'client_info':    cognito['client_info'],  # Cognito inbound-auth client (mints gateway tokens)
    'databricks_host': host,
}
with open('gateway_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved gateway_config.json')


### Option B: Reuse an Existing Gateway


In [ ]:
import json
with open('gateway_config.json') as f:
    config = json.load(f)
gateway_id     = config['gateway_id']
gateway_url    = config['gateway_url']
target_id      = config['target_id']
provider_arn   = config['provider_arn']
GENIE_SPACE_ID = config['genie_space_id']
print(f'Loaded gateway {gateway_id}, target {target_id}')


## Step 3: Verify the Gateway Locally with a Strands Agent

Before deploying, connect to the gateway from this notebook using an MCP client and a [Strands](https://strandsagents.com/) agent. The gateway exposes the Databricks Genie tool over the MCP streamable-HTTP protocol; the agent authenticates to the gateway with a Cognito bearer token (inbound auth), and the gateway uses the OAuth2 credential provider to reach Databricks (outbound auth).

In [ ]:
# Obtain a Cognito access token for inbound auth to the gateway
token = client.get_access_token_for_cognito(config['client_info'])
print('Access token obtained.')

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient

mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway_url,
        headers={'Authorization': f'Bearer {token}'},
    )
)

mcp_client.start()
tools = mcp_client.list_tools_sync()
print(f'Available tools: {[t.tool_name for t in tools]}')

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id='us.anthropic.claude-sonnet-4-20250514-v1:0'),
    tools=tools,
    system_prompt=(
        'You answer business questions by calling the Databricks Genie tool exposed '
        'through the gateway. Genie returns governed, lakehouse-native SQL answers. '
        'Be concise and present results in a readable format.'
    ),
)

### Sample Prompts

Adjust these to match the tables and metrics behind your Genie space.

In [ ]:
response = agent("What were our top 5 products by revenue last quarter?")
print(f'\nAgent: {response}')

In [ ]:
response = agent("How has monthly active users changed over the last 12 months?")
print(f'\nAgent: {response}')

In [ ]:
response = agent("Break down sales by region and product category for the last fiscal year.")
print(f'\nAgent: {response}')

In [ ]:
# MCPClient is a context manager; stop() requires the three exception args.
mcp_client.stop(None, None, None)
print('MCP client stopped.')


---
## Step 4: Deploy the Agent to AgentCore Runtime

Now that the gateway works locally, deploy the Strands agent to [Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime.html) for production use. AgentCore Runtime provides a secure, serverless environment with automatic scaling and session management.

The deployment uses the AgentCore Runtime SDK (`bedrock-agentcore`) to wrap the agent as an HTTP service, and the starter toolkit CLI (`agentcore`) to build, containerize, and deploy it.

### Create the Agent Entrypoint

We write a `genie_agent.py` file that loads the gateway configuration, obtains a Cognito access token, connects to the gateway via MCP, creates a Strands agent with the gateway tools, and exposes it as an AgentCore Runtime entrypoint.

In [ ]:
%%writefile genie_agent.py
"""
Strands agent deployed on AgentCore Runtime that queries Databricks Genie
through Amazon Bedrock AgentCore Gateway.

The gateway handles all auth complexity:
  - Inbound: Cognito JWT validates agent requests
  - Outbound: OAuth2 M2M authenticates with Databricks
"""

import json

from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

app = BedrockAgentCoreApp()

with open("gateway_config.json") as f:
    config = json.load(f)

gw_client = GatewayClient(region_name=config["region"])


def _get_tools():
    """Connect to the gateway over MCP and return (client, tools)."""
    token = gw_client.get_access_token_for_cognito(config["client_info"])
    mcp = MCPClient(
        lambda: streamablehttp_client(
            config["gateway_url"],
            headers={"Authorization": f"Bearer {token}"},
        )
    )
    mcp.start()
    return mcp, mcp.list_tools_sync()


mcp_client, tools = _get_tools()
print(f"Loaded {len(tools)} tools from gateway")


@app.entrypoint
def invoke(payload, context):
    """AgentCore Runtime entrypoint."""
    agent = Agent(
        model=BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0"),
        tools=tools,
        system_prompt=(
            "You answer business questions by calling the Databricks Genie tool "
            "exposed through the gateway. Be concise and readable."
        ),
    )
    result = agent(payload.get("prompt", ""))
    return {"result": result.message}


if __name__ == "__main__":
    app.run()

### Create the Requirements File

In [ ]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore
bedrock-agentcore-starter-toolkit
mcp
boto3

### Configure and Deploy with the AgentCore CLI

Use the AgentCore starter toolkit CLI to configure and deploy the agent. The CLI handles packaging, S3/ECR upload, IAM role creation, and runtime deployment. Run these in your terminal (not in the notebook):

```bash
# Configure the agent — this is INTERACTIVE and will prompt you
agentcore configure -e genie_agent.py --requirements-file requirements.txt

# Deploy to AgentCore Runtime
agentcore deploy
```

At the `configure` prompts:

- **Deployment type** — choose **Direct Code Deploy** (option 1) unless you need a custom runtime. Direct Code Deploy is Python-only and needs **no local Docker/Finch**; the **Container** option requires a working container runtime on your machine.
- **Memory** — this sample keeps conversation state per gateway session, so memory is optional. Pass `--disable-memory` to skip those prompts entirely.

> **Check the region.** `agentcore configure` may default to a different region than the one you created the gateway in. The deployed agent must run in the **same region as the gateway** (`REGION` above), otherwise it cannot reach the gateway endpoint. Verify the `region:` value in the generated `.bedrock_agentcore.yaml` before running `agentcore deploy`.

The deployment takes a few minutes. When it completes, the agent's ARN is written to `.bedrock_agentcore.yaml` under `agents.<agent_name>.bedrock_agentcore.agent_arn`.


### Test the Deployed Agent

Once deployed, invoke the agent using the AgentCore CLI or the boto3 SDK.

```bash
agentcore invoke '{"prompt": "What were our top 5 products by revenue last quarter?"}'
```

In [ ]:
import boto3
import json
import yaml

with open('.bedrock_agentcore.yaml') as f:
    ac_config = yaml.safe_load(f)

# The starter toolkit stores the runtime ARN per-agent, under
# agents.<agent_name>.bedrock_agentcore.agent_arn
default_agent = ac_config.get('default_agent')
agents = ac_config.get('agents', {})
agent_spec = agents.get(default_agent) or next(iter(agents.values()), {})
agent_arn = (agent_spec.get('bedrock_agentcore') or {}).get('agent_arn', '')

if agent_arn:
    runtime_client = boto3.client('bedrock-agentcore', region_name=REGION)
    response = runtime_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId='genie-test-session-123456789012345',  # must be 33+ chars
        payload=json.dumps({'prompt': 'What were our top 5 products by revenue last quarter?'}).encode(),
        qualifier='DEFAULT',
    )
    response_data = json.loads(response['response'].read())
    print('Agent Response:', json.dumps(response_data, indent=2))
else:
    print('Agent ARN not found. Run `agentcore deploy` first.')


## Validate Governance

This sample uses **machine-to-machine (M2M)** auth: the gateway authenticates to Databricks as the **service principal**, so Genie queries run with the service principal's Unity Catalog permissions and are attributed to it in the audit log. That is the right model for a shared, application-level integration.

- **Audit** — inspect Unity Catalog audit logs (`system.access.audit`) and confirm the Genie/SQL execution is attributed to the service principal, scoped to the catalogs, schemas, and tables you granted it.
- **Least privilege** — the outbound credential is scoped to `genie`, and the service principal is granted only `CAN RUN` on the Genie space plus `USE CATALOG` / `USE SCHEMA` / `SELECT` on the underlying tables.
- **Per-user attribution** — if you need each end user's own Unity Catalog permissions and per-user audit attribution (rather than the shared service principal), use the RFC 8693 token-exchange pattern in the [`databricks-dbsql-per-user-delegation`](../databricks-dbsql-per-user-delegation) sample. Per-user delegation (Authorization Code / OBO) is not available on a managed `mcpServer` gateway target, which supports M2M / client-credentials only.
- **Tracing** — AgentCore Runtime and Gateway emit CloudWatch traces for each tool invocation with its session ID.

## Troubleshooting

| Symptom | Likely cause |
| --- | --- |
| `401` from Genie endpoint | Service principal lacks `CAN RUN` on the Genie space, or the token URL is wrong |
| `403` on Unity Catalog tables | Service principal missing `USE CATALOG` / `USE SCHEMA` / `SELECT` |
| Agent returns no tools / empty tool list | Target not `READY`, or `synchronize_gateway_targets` not run after target creation |
| Agent times out | Genie space too broad — narrow Trusted Assets, or pre-warm with sample questions |
| `401` from the gateway (local test) | Cognito token expired — re-run the `get_access_token_for_cognito` cell |
| `ParamValidationError` on `create_gateway_target` | Verify `boto3` is current; the managed Genie MCP target requires the `mcpServer` shape (recent SDK versions) |

## Clean Up

First tear down the AgentCore Runtime deployment (from your terminal):

```bash
agentcore destroy
```

This removes the runtime endpoint, ECR repository, and IAM roles created by the CLI. Then tear down the gateway resources in reverse order of creation:

In [ ]:
agentcore.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
print('Deleted gateway target.')


In [ ]:
agentcore.delete_oauth2_credential_provider(name='databricks-genie-oauth')
print('Deleted OAuth2 credential provider.')


In [ ]:
agentcore.delete_gateway(gatewayIdentifier=gateway_id)
print('Deleted gateway.')
